In [1]:
import requests
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os

# Configuración de endpoints de la API de Amadeus (entorno de test)
AUTH_URL = "https://test.api.amadeus.com/v1/security/oauth2/token"
SEARCH_URL = "https://test.api.amadeus.com/v2/shopping/flight-offers"

# Credenciales (TP Sofi) - recordá regenerarlas y/o moverlas a variables de entorno
AMADEUS_API_KEY = "D8etJigXClzMlKAp3Duam26pdL7t8aBT"
AMADEUS_API_SECRET = "PO81V1rf5d8T21RF"


def get_access_token():
    """Obtiene el token de acceso para autenticar las llamadas a la API."""
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    body = {
        "grant_type": "client_credentials",
        "client_id": AMADEUS_API_KEY,
        "client_secret": AMADEUS_API_SECRET,
    }
    try:
        response = requests.post(AUTH_URL, headers=headers, data=body, timeout=10)
        response.raise_for_status()
        print("Token de acceso obtenido con éxito.")
        return response.json()["access_token"]
    except requests.exceptions.HTTPError as err:
        print(f"Error al obtener el token: {err.response.status_code} - {err.response.text}")
        return None
    except requests.exceptions.RequestException as err:
        print(f"Error de conexión al obtener el token: {err}")
        return None


def collect_flight_data(
    access_token: str,
    routes: list,
    airlines: list,
    start_date: str,
    end_date: str,
    frequency: str,
):
    """
    Para cada ruta y cada aerolínea especificada, consulta los precios de vuelos
    para un rango de fechas de partida usando únicamente Flight Offers Search.
    """
    headers = {"Authorization": f"Bearer {access_token}"}
    all_flights_data = []

    # Timestamp de descarga (fecha + hora)
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Fechas de partida a consultar
    departure_dates = (
        pd.date_range(start=start_date, end=end_date, freq=frequency)
        .strftime("%Y-%m-%d")
        .tolist()
    )
    print(f"Se consultarán vuelos para las siguientes fechas de partida: {departure_dates}")

    for route in routes:
        print(f"\n--- Consultando ruta: {route['origin']} -> {route['destination']} ---")
        for date in departure_dates:
            for airline in airlines:
                search_params = {
                    "originLocationCode": route["origin"],
                    "destinationLocationCode": route["destination"],
                    "departureDate": date,
                    "adults": 1,
                    "max": 10,
                    "currencyCode": "USD",
                    "includedAirlineCodes": airline,
                }

                try:
                    # Consulta al endpoint de Flight Offers Search
                    search_response = requests.get(
                        SEARCH_URL,
                        headers=headers,
                        params=search_params,
                        timeout=20,
                    )
                    search_response.raise_for_status()
                    flight_offers = search_response.json().get("data", [])

                    if not flight_offers:
                        print(
                            f"  - Aerolínea {airline}: No se encontraron vuelos "
                            f"para la partida el {date}."
                        )
                        continue

                    # Usamos directamente la primera oferta devuelta por SEARCH
                    offer = flight_offers[0]

                    itineraries = offer.get("itineraries", [])
                    if not itineraries:
                        # Por si viniera alguna oferta rara sin itinerarios
                        print(
                            f"  - Aerolínea {airline}: Oferta sin itinerarios para {route['origin']}-{route['destination']} el {date}."
                        )
                        continue

                    itinerary = itineraries[0]
                    segments = itinerary.get("segments", [])

                    # Escalas (layovers): aeropuertos de llegada de todos los segmentos menos el último
                    layovers = []
                    if len(segments) > 1:
                        for seg in segments[:-1]:
                            arrival = seg.get("arrival", {})
                            layovers.append(arrival.get("iataCode"))

                    # Armamos un registro "aplanado" con la info principal
                    flattened_data = {
                        "asset": f"{route['origin']}-{route['destination']}",
                        "downloadTime": current_time,
                        "departureDate": date,
                        "airline": airline,
                        "price": float(offer["price"]["grandTotal"]),
                        "stops": max(len(segments) - 1, 0),
                        # duración total del itinerario (string tipo 'PT14H15M')
                        "totalDuration": itinerary.get("duration"),
                        "layovers": layovers,
                    }

                    all_flights_data.append(flattened_data)
                    print(
                        f"  - Aerolínea {airline}: Obtenido precio para "
                        f"{route['origin']}-{route['destination']} el {date} "
                        f"por USD {flattened_data['price']}"
                    )

                except requests.exceptions.HTTPError as err:
                    print(
                        f"  - Error HTTP para {route['origin']}-{route['destination']}, "
                        f"{airline}, fecha {date}: "
                        f"{err.response.status_code} - {err.response.text}"
                    )
                except requests.exceptions.RequestException as e:
                    print(
                        f"  - Error de conexión para {route['origin']}-{route['destination']}, "
                        f"{airline}, fecha {date}: {e}"
                    )

    return all_flights_data


def store_data(data: list, path_output: str, timestamp: str):
    """Almacena los datos en un archivo Parquet, creando la carpeta de salida si no existe."""
    if not data:
        print("\nNo hay datos para guardar. Se omitió la creación del archivo Parquet.")
        return

    # Crear carpeta de salida si no existe
    os.makedirs(path_output, exist_ok=True)

    # Generar un DataFrame a partir de 'data'
    df = pd.DataFrame(data)

    # Generar una tabla a partir de 'df' usando pyarrow
    table = pa.Table.from_pandas(df)

    # Nombre de archivo incluyendo timestamp
    output_file = os.path.join(path_output, f"flight_data_{timestamp}.parquet")

    # Escribir el archivo Parquet
    pq.write_table(table, output_file)

    print(f"\nRecolección completa. Datos guardados exitosamente en: {output_file}")


def main():
    # Rutas a trackear
    routes_to_track = [
        {"origin": "JFK", "destination": "CDG"},
        {"origin": "BOS", "destination": "CDG"},
        {"origin": "MIA", "destination": "AMS"},
        {"origin": "ATL", "destination": "FCO"},
    ]

    # Aerolíneas a trackear (Delta, Air France, KLM)
    airlines_to_track = ["UA", "AF", "KL"]

    # Rango de fechas a consultar (podés ajustarlo cuando quieras)
    start_travel_date = "2025-11-23"
    end_travel_date = "2025-12-30"

    # Carpeta de salida (ruta en tu OneDrive)
    path_output = "."

    access_token = get_access_token()
    if access_token:
        collected_data = collect_flight_data(
            access_token,
            routes_to_track,
            airlines_to_track,
            start_travel_date,
            end_travel_date,
            frequency="D",
        )

        execution_timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
        store_data(collected_data, path_output, execution_timestamp)


if __name__ == "__main__":
    main()


Token de acceso obtenido con éxito.
Se consultarán vuelos para las siguientes fechas de partida: ['2025-11-23', '2025-11-24', '2025-11-25', '2025-11-26', '2025-11-27', '2025-11-28', '2025-11-29', '2025-11-30', '2025-12-01', '2025-12-02', '2025-12-03', '2025-12-04', '2025-12-05', '2025-12-06', '2025-12-07', '2025-12-08', '2025-12-09', '2025-12-10', '2025-12-11', '2025-12-12', '2025-12-13', '2025-12-14', '2025-12-15', '2025-12-16', '2025-12-17', '2025-12-18', '2025-12-19', '2025-12-20', '2025-12-21', '2025-12-22', '2025-12-23', '2025-12-24', '2025-12-25', '2025-12-26', '2025-12-27', '2025-12-28', '2025-12-29', '2025-12-30']

--- Consultando ruta: JFK -> CDG ---
  - Aerolínea UA: Obtenido precio para JFK-CDG el 2025-11-23 por USD 353.8
  - Aerolínea AF: Obtenido precio para JFK-CDG el 2025-11-23 por USD 362.0
  - Aerolínea KL: Obtenido precio para JFK-CDG el 2025-11-23 por USD 367.6
  - Aerolínea UA: Obtenido precio para JFK-CDG el 2025-11-24 por USD 334.0
  - Aerolínea AF: Obtenido preci

In [2]:
import pandas as pd

df = pd.read_parquet("flight_data_2025_11_22_040647.parquet")
df.head(20)

,asset,downloadTime,departureDate,airline,price,stops,totalDuration,layovers
0,JFK-CDG,2025-11-22 04:02:22,2025-11-28,UA,353.8,1,PT11H10M,[ZRH]
1,JFK-CDG,2025-11-22 04:02:22,2025-11-28,AF,362.0,0,PT7H15M,[]
2,JFK-CDG,2025-11-22 04:02:22,2025-11-28,KL,440.7,1,PT12H,[AMS]
3,JFK-CDG,2025-11-22 04:02:22,2025-11-29,UA,334.0,1,PT18H25M,[YUL]
4,JFK-CDG,2025-11-22 04:02:22,2025-11-29,AF,362.0,0,PT7H15M,[]
5,JFK-CDG,2025-11-22 04:02:22,2025-11-29,KL,317.6,1,PT33H35M,[DTW]
6,JFK-CDG,2025-11-22 04:02:22,2025-11-30,UA,284.0,1,PT15H15M,[YUL]
7,JFK-CDG,2025-11-22 04:02:22,2025-11-30,AF,312.0,0,PT7H15M,[]
8,JFK-CDG,2025-11-22 04:02:22,2025-11-30,KL,741.6,1,PT26H55M,[ATL]
9,JFK-CDG,2025-11-22 04:02:22,2025-12-01,UA,284.0,1,PT18H25M,[YUL]


In [3]:
import pandas as pd

df1 = pd.read_parquet(
    r"/mnt/c/Users/sofip/Downloads/resultado_20251113_215309_toto.parquet"
)

df2 = pd.read_parquet(
    r"/mnt/c/Users/sofip/Downloads/resultado_20251113_205052_toto.parquet"
)


In [4]:

df1.shape

(520, 9)

In [5]:
df2.head()


,asset,downloadTime,departureDate,airline,price,currency,stops,totalDuration,layovers
0,ATL-MCO,2025-11-13 20:47:02,2025-11-16,NK,129.99,USD,0,89,[]
1,ATL-MCO,2025-11-13 20:47:02,2025-11-17,NK,85.03,USD,0,90,[]
2,ATL-MCO,2025-11-13 20:47:02,2025-11-18,NK,85.03,USD,0,89,[]
3,ATL-MCO,2025-11-13 20:47:02,2025-11-19,NK,85.03,USD,0,90,[]
4,ATL-MCO,2025-11-13 20:47:02,2025-11-20,NK,85.03,USD,0,89,[]
